# Autoencoder on IFCNet: BIM object similarity
Team group2_hackathon · AI Methods I hackathon · Monday

## 0. Setup

In [2]:
from pathlib import Path
import random
import numpy as np
import torch
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. The data

In [ ]:
ROOT = Path("../../data")      # data/ is two levels up from notebook/
IMG = 64                       # image size in pixels
VIEWS = ["0", "3", "6"]        # 3 of the 12 rendered views per object
MAX_OBJ_PER_CLASS = 300        # small first run

# 1) Find all images. Path pattern: data/<IfcClass>/<train|test>/<object_id>.<view>.png
pngs = sorted(ROOT.rglob("*.png"))
print(len(pngs), "png files")

def info(p):
    rel = p.relative_to(ROOT).parts
    cls, split = rel[0], rel[1]
    obj, view = p.stem.rsplit(".", 1)
    return cls, split, obj, view

# 2) Group the chosen views of each object
groups = {}
for p in pngs:
    cls, split, obj, view = info(p)
    if view in VIEWS:
        groups.setdefault((cls, split, obj), []).append(p)

classes = sorted({k[0] for k in groups})
print(len(classes), "classes:", classes)

# 3) Load as 64x64 greyscale, values 0-1, object bright on dark background
def load(split):
    X, y, ids = [], [], []
    for c_i, cls in enumerate(classes):
        objs = [k for k in groups if k[0] == cls and k[1] == split]
        random.shuffle(objs)
        for k in objs[:MAX_OBJ_PER_CLASS]:
            for p in groups[k]:
                im = Image.open(p).convert("L").resize((IMG, IMG))
                X.append(1 - np.asarray(im, np.float32) / 255)
                y.append(c_i)
                ids.append(k[2])
    return torch.tensor(np.stack(X))[:, None], torch.tensor(y), ids

X_trval, y_trval, ids_trval = load("train")
X_test, y_test, _ = load("test")

# 4) Validation split BY OBJECT, so views of one object never sit in both train and val
uniq = sorted(set(ids_trval))
random.shuffle(uniq)
val_ids = set(uniq[: len(uniq) // 7])
m = torch.tensor([i in val_ids for i in ids_trval])
X_train, y_train = X_trval[~m], y_trval[~m]
X_val, y_val = X_trval[m], y_trval[m]

print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)

95160 png files
20 classes: ['IfcAirTerminal', 'IfcBeam', 'IfcCableCarrierFitting', 'IfcCableCarrierSegment', 'IfcDoor', 'IfcDuctFitting', 'IfcDuctSegment', 'IfcFurniture', 'IfcLamp', 'IfcOutlet', 'IfcPipeFitting', 'IfcPipeSegment', 'IfcPlate', 'IfcRailing', 'IfcSanitaryTerminal', 'IfcSlab', 'IfcSpaceHeater', 'IfcStair', 'IfcValve', 'IfcWall']
